# LegalGPT: sentence-role classifier, round 2 (longer training + public-corpus transfer)

1. **Longer training** of the deployed setup: 10 epochs, 384 tokens, lower learning rate.
2. **Transfer:** fine-tune InLegalBERT on the public *InRhetoricalRoles* corpus (Kalamkar et al.,
   LREC 2022), mapped to our 5 roles, then cross-validate on our 36 judgments from that checkpoint.
3. The BiLSTM sequence model on top of the transferred encoder, also pretrained on the public judgments.

Same 5 folds as round 1; every held-out prediction is saved, the ensembles and tests are computed afterwards.

**Before you start:**
- On Hugging Face, open https://huggingface.co/datasets/opennyaiorg/InRhetoricalRoles while logged
  in and accept its access conditions (the dataset is gated; licence CC BY-SA 4.0 / Apache-2.0).
- Create a *read* token at https://huggingface.co/settings/tokens. You paste it into the login box
  below; it stays in Colab.
- Runtime → Change runtime type → **L4 GPU**. Then run the cells in order (or Run all; the
  upload and login cells wait for you).

About 45–60 minutes on an L4. At the end `rrl_results_2.zip` downloads; send it back.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Upload the package and install

In [ ]:
import os, zipfile
from google.colab import files
os.chdir('/content')
if not os.path.exists('/content/legaltech/backend/app'):
    up = files.upload()  # choose legalgpt_rrl.zip
    zipfile.ZipFile(next(iter(up))).extractall('/content/legaltech')
os.chdir('/content/legaltech')
!pip -q install transformers==5.18.0 pydantic-settings==2.15.0 python-dotenv==1.2.4
!pip -q uninstall -y torchao

## 2. Download the public corpus (needs the accepted terms and your token)

In [ ]:
from huggingface_hub import login, hf_hub_download
login()  # paste your read token in the box
paths = {s: hf_hub_download('opennyaiorg/InRhetoricalRoles', f'{s}.json', repo_type='dataset', local_dir='/content/inrr')
         for s in ('train', 'dev')}
print(paths)

In [ ]:
# map the 13 public roles to our 5 and drop any public judgment that overlaps ours
!python backend/scripts/build_rrl_public.py --train /content/inrr/train.json --dev /content/inrr/dev.json --version v0.3

## 3. Stage A: InLegalBERT on the public corpus

In [ ]:
rc = os.system('python backend/scripts/pretrain_rrl_public.py --epochs 2 --bf16 > data/models/pretrain_public.log 2>&1')
os.system('grep -E "public corpus|macro-F1|saved" data/models/pretrain_public.log | tail -4')
import glob
CKPT = sorted(glob.glob('data/models/rrl_public/inrr-*'))[-1]
print('exit code', rc, '| checkpoint', CKPT)

## 4. Cross-validation on our judgments (same folds as round 1)

In [ ]:
VERSION = 'v0.3'
BASE = f'python backend/scripts/train_rrl_classifier.py --version {VERSION} --freeze-layers 0 --bf16 --cv 5 --save-predictions --context'
RUNS = {
    'long':     '--max-length 384 --epochs 10 --lr 2e-5',                       # 1. longer training, from InLegalBERT
    'pub-full': f'--max-length 256 --epochs 6 --model {CKPT}',                  # 2. from the public-corpus checkpoint
    'pub-long': f'--max-length 384 --epochs 10 --lr 2e-5 --model {CKPT}',       # 1 + 2
}
for tag, extra in RUNS.items():
    print(f'=== {tag} ===', flush=True)
    rc = os.system(f'{BASE} {extra} --tag {tag} > data/models/{tag}.log 2>&1')
    os.system(f'grep -E "mean over" data/models/{tag}.log | tail -1')
    print('exit code', rc, flush=True)

## 5. Sequence models

In [ ]:
SEQ = f'python backend/scripts/rrl_sequence_model.py --version {VERSION} --folds-from "data/models/rrl_cv/{VERSION}-cv5-pub-full-*.json"'
for tag, extra in {
    'seq-pubbilstm': '--pretrain-public data/rrl_public',                         # BiLSTM pretrained on public judgments
    'seq-pubenc':    f'--encoder {CKPT} --pretrain-public data/rrl_public',       # + vectors from the transferred encoder
}.items():
    print(f'=== {tag} ===', flush=True)
    rc = os.system(f'{SEQ} {extra} --tag {tag} > data/models/{tag}.log 2>&1')
    os.system(f'grep -E "pretrained|sequence:" data/models/{tag}.log | tail -2')
    print('exit code', rc, flush=True)

## 6. Download the results

In [ ]:
with zipfile.ZipFile('/content/rrl_results_2.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in (glob.glob('data/models/rrl_cv/*.json') + glob.glob('data/models/*.log')
              + glob.glob('data/models/rrl_public/*/metrics.json') + glob.glob('data/models/rrl_public/*/rrl_meta.json')):
        z.write(f)
print(os.path.getsize('/content/rrl_results_2.zip') // 1024, 'KB')
files.download('/content/rrl_results_2.zip')

## 7. (Optional) keep the public-corpus checkpoint

Only needed if the transfer model ends up in the app. Copies it (~0.4 GB) to your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.system(f'mkdir -p /content/drive/MyDrive/legalgpt && cp -r {CKPT} /content/drive/MyDrive/legalgpt/')
print('copied', CKPT)